In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl.extras.vllm_client import VLLMClient
from typing import Optional
from vllm import RequestOutput

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
#model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", device_map="auto", torch_dtype=torch.bfloat16)

messages = [
    {"role": "system", "content": "You are a friendly chatbot who always responds in the style of a pirate",},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
 ]
tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
print(tokenizer.decode(tokenized_chat[0]))

<｜begin▁of▁sentence｜>You are a friendly chatbot who always responds in the style of a pirate<｜User｜>How many helicopters can a human eat in one sitting?<｜Assistant｜><think>



In [ ]:
RequestOutput(
    prompt_token_ids= tokenized_chat[0],
    
)

In [8]:
tokenized_chat

tensor([[151646,   2610,    525,    264,  11657,   6236,   6331,    879,   2677,
          30580,    304,    279,   1707,    315,    264,  53966, 151644,   4340,
           1657,  58332,    646,    264,   3738,   8180,    304,    825,  11699,
             30, 151645, 151648,    198]])

In [3]:
class VLLMClient_new(VLLMClient):
    def generate(
        self,
        prompts: list[str],
        n: int = 1,
        repetition_penalty: float = 1.0,
        temperature: float = 1.0,
        top_p: float = 1.0,
        top_k: int = -1,
        min_p: float = 0.0,
        max_tokens: int = 16,
        guided_decoding_regex: Optional[str] = None,
    ) -> list[list[str]]:
        url = f"http://{self.host}:{self.server_port}/generate/"
        response = self.session.post(
            url,
            json={
                "prompts": prompts,
                "n": n,
                "repetition_penalty": repetition_penalty,
                "temperature": temperature,
                "top_p": top_p,
                "top_k": top_k,
                "min_p": min_p,
                "max_tokens": max_tokens,
                "guided_decoding_regex": guided_decoding_regex,
            },
        )
        if response.status_code == 200:
            return response.json()#["completion_ids"]
        else:
            raise Exception(f"Request failed: {response.status_code}, {response.text}")


In [4]:
client = VLLMClient_new()

INFO 03-31 20:54:38 [utils.py:925] Found nccl from library libnccl.so.2
INFO 03-31 20:54:38 [pynccl.py:69] vLLM is using nccl==2.26.2


In [5]:
text = tokenizer.decode(tokenized_chat[0])

In [6]:
outputs = client.generate([text], max_tokens=1024) 
print(tokenizer.decode(outputs[0]))

KeyError: 0

In [7]:
outputs

{'completion_ids': [[32313,
   11,
   279,
   1196,
   374,
   10161,
   1246,
   1657,
   58332,
   264,
   3738,
   646,
   8180,
   304,
   825,
   11699,
   11,
   714,
   807,
   2299,
   22331,
   264,
   53966,
   56589,
   2033,
   13,
   6771,
   752,
   1744,
   911,
   429,
   382,
   5338,
   11,
   279,
   15145,
   374,
   9355,
   279,
   3738,
   11,
   323,
   23988,
   264,
   42749,
   374,
   2155,
   13,
   576,
   3681,
   4226,
   572,
   1101,
   264,
   21646,
   11,
   892,
   374,
   1661,
   369,
   1351,
   2096,
   1403,
   11,
   714,
   7196,
   429,
   594,
   264,
   2699,
   9575,
   382,
   40,
   1265,
   14198,
   279,
   3405,
   803,
   40340,
   13,
   1084,
   594,
   264,
   111373,
   6001,
   1681,
   11,
   773,
   358,
   1184,
   311,
   5889,
   3381,
   3641,
   2041,
   916,
   5342,
   10732,
   13,
   18765,
   34171,
   2176,
   1410,
   1281,
   432,
   803,
   22414,
   382,
   18592,
   38270,
   4436,
   944,
   264,
   91533,
 

In [4]:
def extract_after_last_assistant(text: str) -> str:
    marker = "<｜Assistant｜>"
    eos_marker = "<｜end▁of▁sentence｜>"
    last_occurrence = text.rfind(marker)
    if last_occurrence == -1:
        return ""  # Return empty string if marker is not found
    result = text[last_occurrence + len(marker):].strip()
    if result.endswith(eos_marker):
        result = result[:-len(eos_marker)].strip()
    return result

message_ = extract_after_last_assistant(tokenizer.decode(outputs[0]))
messages.append({"role":"assistant", "content": message_})

In [5]:
messages

[{'role': 'system',
  'content': 'You are a friendly chatbot who always responds in the style of a pirate'},
 {'role': 'user',
  'content': 'How many helicopters can a human eat in one sitting?'},
 {'role': 'assistant',
  'content': "<think>\nAlright, so I'm trying to figure out how many helicopters a human can eat in one sitting. Hmm, that's an interesting question. First off, I know that helicopters are expensive and they're used for various purposes, like taking off, landing, and even for cargo. But can a human even think about eating them?\n\nMaybe the question is more about comparing the costs or the resources needed to operate a helicopter versus a human. Or perhaps it's a playful way to ask how much effort or resources a human would need to handle something as big and expensive as a helicopter.\n\nI should consider the context in which the question is asked. If it's a real-world scenario, maybe it's about comparing the cost of operating a helicopter versus a human, but that does

In [6]:
tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
print(tokenizer.decode(tokenized_chat[0]))

<｜begin▁of▁sentence｜>You are a friendly chatbot who always responds in the style of a pirate<｜User｜>How many helicopters can a human eat in one sitting?<｜Assistant｜>

The question posed is more of a metaphor or joke than a literal one. It's a playful way to explore the costs or resources associated with operating a helicopter versus a human, but in reality, it's not feasible to compare these two due to their vastly different capabilities and purposes. Therefore, the answer is that the question is more of a metaphor or joke, and the intent likely lies elsewhere.<｜end▁of▁sentence｜><｜Assistant｜><think>

